# 01 — Data Loading, Cleaning & Merging

**Purpose**: Load IMDb TSV files and TMDb Kaggle dataset, clean, merge, and save unified dataframe.

**Outputs**:
- `data/movies_clean.parquet` — merged movie dataframe
- Console output: shapes, dtypes, sample rows, null counts at each step

**Key decisions** (see wiki Decision Log):
- D007: TMDb data from Kaggle alanvourch dataset (not API)
- D010: Min vote threshold configurable, default 500

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Paths
SOURCE_DIR = os.path.join('..', 'source_data')
DATA_DIR = os.path.join('..', 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# Config (D010: configurable min vote threshold)
MIN_VOTE_THRESHOLD = 500
CURRENT_YEAR = 2025  # Filter out future releases

print('Config:')
print(f'  MIN_VOTE_THRESHOLD = {MIN_VOTE_THRESHOLD}')
print(f'  CURRENT_YEAR = {CURRENT_YEAR}')
print(f'  SOURCE_DIR = {os.path.abspath(SOURCE_DIR)}')
print(f'  DATA_DIR = {os.path.abspath(DATA_DIR)}')

## 1. Load IMDb Datasets

Loading 5 TSV files from IMDb bulk data. Key notes:
- `\N` = null in IMDb format → replace with NaN
- Filter `titleType == 'movie'` (exclude shorts, TV, etc.)
- Filter `startYear <= CURRENT_YEAR` (exclude future releases)

In [ ]:
def load_imdb_tsv(filename, source_dir=SOURCE_DIR):
    """Load an IMDb TSV.gz file, replace \\N with NaN."""
    path = os.path.join(source_dir, filename)
    print(f'Loading {filename}...')
    df = pd.read_csv(path, sep='\t', compression='gzip', 
                     na_values='\\N', low_memory=False)
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Nulls:\n{df.isnull().sum()}')
    print(f'  Sample:')
    display(df.head(3))
    return df

In [ ]:
# Load title.basics — movie metadata
title_basics = load_imdb_tsv('title.basics.tsv.gz')

In [ ]:
# Filter to movies only and exclude future releases
print(f'Before filter: {len(title_basics):,} rows')
print(f'Title types: {title_basics["titleType"].value_counts().to_dict()}')

title_basics = title_basics[
    (title_basics['titleType'] == 'movie') & 
    (title_basics['startYear'] <= CURRENT_YEAR)
].copy()

print(f'After filter (movie + startYear <= {CURRENT_YEAR}): {len(title_basics):,} rows')
print(f'Start year range: {title_basics["startYear"].min():.0f} - {title_basics["startYear"].max():.0f}')
display(title_basics.head(3))

In [ ]:
# Load title.ratings — ratings and vote counts
title_ratings = load_imdb_tsv('title.ratings.tsv.gz')

In [ ]:
# Load title.crew — director and writer IDs
title_crew = load_imdb_tsv('title.crew.tsv.gz')

In [ ]:
# Load title.principals — cast/crew per title (this is the largest file ~788MB)
# Only load actor/actress category to save memory (D008)
print('Loading title.principals.tsv.gz (filtering to actor/actress only)...')
title_principals = pd.read_csv(
    os.path.join(SOURCE_DIR, 'title.principals.tsv.gz'),
    sep='\t', compression='gzip', na_values='\\N', low_memory=False
)
print(f'  Raw shape: {title_principals.shape}')
print(f'  Categories: {title_principals["category"].value_counts().to_dict()}')

# Filter to actor/actress only
title_principals = title_principals[
    title_principals['category'].isin(['actor', 'actress'])
].copy()
print(f'  After actor/actress filter: {title_principals.shape}')
display(title_principals.head(3))

In [ ]:
# Load name.basics — person metadata
name_basics = load_imdb_tsv('name.basics.tsv.gz')

## 2. Load TMDb Kaggle Dataset (D007)

alanvourch's "Ultimate 1M Movies Dataset (TMDB + IMDb)" from Kaggle.
- Has `imdb_id` for joining with IMDb data
- Provides budget, revenue, cast, director, genres
- 1,261,171 rows, 30 columns

In [ ]:
# Load TMDb CSV from zip
tmdb_zip = os.path.join(SOURCE_DIR, 'alanvourch-tmdb-movies-daily-updates.zip')
print(f'Loading TMDb from {tmdb_zip}...')

with zipfile.ZipFile(tmdb_zip) as z:
    with z.open('TMDB_all_movies.csv') as f:
        tmdb = pd.read_csv(f, low_memory=False)

print(f'Shape: {tmdb.shape}')
print(f'Columns: {list(tmdb.columns)}')
print(f'\nNull counts:')
print(tmdb.isnull().sum())
print(f'\nSample:')
display(tmdb.head(3))

In [ ]:
# Check TMDb data quality (F003: budget/revenue sparsity)
print('TMDb Data Quality:')
print(f'  Total rows: {len(tmdb):,}')
print(f'  Has imdb_id: {tmdb["imdb_id"].notna().sum():,} ({tmdb["imdb_id"].notna().mean()*100:.1f}%)')
print(f'  Has budget > 0: {(tmdb["budget"] > 0).sum():,} ({(tmdb["budget"] > 0).mean()*100:.1f}%)')
print(f'  Has revenue > 0: {(tmdb["revenue"] > 0).sum():,} ({(tmdb["revenue"] > 0).mean()*100:.1f}%)')
print(f'  Has both budget & revenue > 0: {((tmdb["budget"] > 0) & (tmdb["revenue"] > 0)).sum():,} ({((tmdb["budget"] > 0) & (tmdb["revenue"] > 0)).mean()*100:.1f}%)')
print(f'  Has IMDb votes > 0: {(tmdb["imdb_votes"] > 0).sum():,} ({(tmdb["imdb_votes"] > 0).mean()*100:.1f}%)')

## 3. Merge IMDb Data

Merge order:
1. `title_basics` + `title_ratings` → `movies` (on `tconst`)
2. `movies` + `title_crew` → add director/writer IDs
3. `movies` + `title_principals` → add cast info
4. `movies` + `name_basics` → add person names

In [ ]:
# Step 1: Merge basics + ratings
movies = title_basics.merge(title_ratings, on='tconst', how='left')
print(f'After basics + ratings merge: {len(movies):,} rows')
print(f'  Has rating: {movies["averageRating"].notna().sum():,}')
print(f'  Has numVotes: {movies["numVotes"].notna().sum():,}')
display(movies.head(3))

In [ ]:
# Step 2: Merge with crew (directors)
movies = movies.merge(title_crew, on='tconst', how='left')
print(f'After crew merge: {len(movies):,} rows')
print(f'  Has directors: {movies["directors"].notna().sum():,}')

# Get primary director name
# directors field is comma-separated nconst values; take first one
movies['primary_director_nconst'] = movies['directors'].str.split(',').str[0]

# Merge with name_basics to get director name
director_names = name_basics[['nconst', 'primaryName']].rename(
    columns={'nconst': 'primary_director_nconst', 'primaryName': 'director_name'}
)
movies = movies.merge(director_names, on='primary_director_nconst', how='left')
print(f'  Has director name: {movies["director_name"].notna().sum():,}')
display(movies[['tconst', 'primaryTitle', 'director_name']].head(3))

## 4. Merge with TMDb Data (D007)

Join on `tconst` = `imdb_id` to add budget, revenue, and other TMDb fields.

In [ ]:
# Prepare TMDb for merge — rename imdb_id to tconst
tmdb_merge = tmdb.rename(columns={'imdb_id': 'tconst'})

# Select key columns from TMDb to avoid column conflicts
tmdb_cols = ['tconst', 'budget', 'revenue', 'popularity', 'production_companies', 
             'production_countries', 'certification_us', 'keywords']
tmdb_merge = tmdb_merge[tmdb_cols].copy()

# Drop duplicates on tconst (keep first)
tmdb_merge = tmdb_merge.drop_duplicates(subset='tconst', keep='first')
print(f'TMDb unique tconst values: {len(tmdb_merge):,}')

# Merge
movies = movies.merge(tmdb_merge, on='tconst', how='left')
print(f'After TMDb merge: {len(movies):,} rows')
print(f'  Has budget > 0: {(movies["budget"] > 0).sum():,}')
print(f'  Has revenue > 0: {(movies["revenue"] > 0).sum():,}')
print(f'  Has both: {((movies["budget"] > 0) & (movies["revenue"] > 0)).sum():,}')
display(movies[['tconst', 'primaryTitle', 'budget', 'revenue']].head(3))

## 5. Apply Min Vote Threshold (D010)

Filter movies with too few votes — unreliable ratings. Threshold is configurable.

In [ ]:
print(f'Before vote filter: {len(movies):,} rows')
print(f'  Has numVotes: {movies["numVotes"].notna().sum():,}')
print(f'  numVotes >= {MIN_VOTE_THRESHOLD}: {(movies["numVotes"] >= MIN_VOTE_THRESHOLD).sum():,}')

movies_filtered = movies[movies['numVotes'] >= MIN_VOTE_THRESHOLD].copy()
print(f'\nAfter vote filter (numVotes >= {MIN_VOTE_THRESHOLD}): {len(movies_filtered):,} rows')
print(f'  Year range: {movies_filtered["startYear"].min():.0f} - {movies_filtered["startYear"].max():.0f}')
print(f'  Rating range: {movies_filtered["averageRating"].min():.1f} - {movies_filtered["averageRating"].max():.1f}')
print(f'  numVotes range: {movies_filtered["numVotes"].min():.0f} - {movies_filtered["numVotes"].max():.0f}')

## 6. Compute Actor Popularity (D008, D009)

Formula: `Actor Popularity = SUM( (1/log(ordering+1)) × numVotes × averageRating )`

Only for movies in our filtered dataset.

In [ ]:
# Get tconst set for filtered movies
filtered_tconsts = set(movies_filtered['tconst'])

# Filter principals to our movies
principals_filtered = title_principals[
    title_principals['tconst'].isin(filtered_tconsts)
].copy()

print(f'Principals for filtered movies: {len(principals_filtered):,} rows')
print(f'Unique actors: {principals_filtered["nconst"].nunique():,}')

# Merge with ratings
principals_with_ratings = principals_filtered.merge(
    title_ratings, on='tconst', how='left'
)

# Compute billing weight (D009: 1/log(ordering+1))
principals_with_ratings['billing_weight'] = 1.0 / np.log(
    principals_with_ratings['ordering'].astype(float) + 1
)

# Compute weighted score per appearance
principals_with_ratings['weighted_score'] = (
    principals_with_ratings['billing_weight'] * 
    principals_with_ratings['numVotes'].fillna(0) * 
    principals_with_ratings['averageRating'].fillna(0)
)

# Aggregate by actor
actor_popularity = principals_with_ratings.groupby('nconst').agg(
    total_score=('weighted_score', 'sum'),
    num_movies=('tconst', 'nunique'),
    total_votes=('numVotes', 'sum')
).reset_index()

# Add actor names
actor_names = name_basics[['nconst', 'primaryName']].drop_duplicates()
actor_popularity = actor_popularity.merge(actor_names, on='nconst', how='left')

# Sort by popularity
actor_popularity = actor_popularity.sort_values('total_score', ascending=False)

print(f'\nTop 20 Most Popular Actors (across all years in dataset):')
display(actor_popularity.head(20)[['primaryName', 'total_score', 'num_movies', 'total_votes']])

## 7. Compute Director Performance

Director historical performance = average weighted rating of their past films.

In [ ]:
# Director stats from filtered movies
director_stats = movies_filtered.groupby('director_name').agg(
    num_movies=('tconst', 'nunique'),
    avg_rating=('averageRating', 'mean'),
    avg_votes=('numVotes', 'mean'),
    total_votes=('numVotes', 'sum')
).reset_index()

director_stats = director_stats.sort_values('total_votes', ascending=False)

print(f'Top 20 Directors by Total Audience Reach:')
display(director_stats.head(20))

## 8. Final Dataframe & Save

Save cleaned and merged dataframe to parquet for use in EDA notebook.

In [ ]:
# Final summary
print('='*60)
print('FINAL DATAFRAME SUMMARY')
print('='*60)
print(f'Shape: {movies_filtered.shape}')
print(f'\nColumns: {list(movies_filtered.columns)}')
print(f'\nDtypes:\n{movies_filtered.dtypes}')
print(f'\nNull counts:\n{movies_filtered.isnull().sum()}')
print(f'\nSample (top rated):')
display(movies_filtered.nlargest(5, 'averageRating')[
    ['tconst', 'primaryTitle', 'startYear', 'averageRating', 'numVotes', 'director_name', 'budget', 'revenue']
])

In [ ]:
# Save to parquet
output_path = os.path.join(DATA_DIR, 'movies_clean.parquet')
movies_filtered.to_parquet(output_path, index=False)
print(f'Saved to {os.path.abspath(output_path)}')
print(f'File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB')

# Also save actor popularity
actor_path = os.path.join(DATA_DIR, 'actor_popularity.parquet')
actor_popularity.to_parquet(actor_path, index=False)
print(f'Saved actor popularity to {os.path.abspath(actor_path)}')

# Also save director stats
director_path = os.path.join(DATA_DIR, 'director_stats.parquet')
director_stats.to_parquet(director_path, index=False)
print(f'Saved director stats to {os.path.abspath(director_path)}')

## 9. Quick 2023 Snapshot

Preview 2023 movies for EDA Q1 and Q2.

In [ ]:
# 2023 movies
movies_2023 = movies_filtered[movies_filtered['startYear'] == 2023].copy()
print(f'2023 movies with >= {MIN_VOTE_THRESHOLD} votes: {len(movies_2023):,}')

# Top 10 by rating
print(f'\nTop 10 Highest Rated 2023 Movies:')
display(movies_2023.nlargest(10, 'averageRating')[
    ['primaryTitle', 'averageRating', 'numVotes', 'director_name']
])

# Top 10 by votes (most popular)
print(f'\nTop 10 Most Voted 2023 Movies:')
display(movies_2023.nlargest(10, 'numVotes')[
    ['primaryTitle', 'averageRating', 'numVotes', 'director_name']
])

In [ ]:
# 2023 actor popularity
principals_2023 = title_principals[
    title_principals['tconst'].isin(set(movies_2023['tconst']))
].merge(title_ratings, on='tconst', how='left')

principals_2023['billing_weight'] = 1.0 / np.log(principals_2023['ordering'].astype(float) + 1)
principals_2023['weighted_score'] = (
    principals_2023['billing_weight'] * 
    principals_2023['numVotes'].fillna(0) * 
    principals_2023['averageRating'].fillna(0)
)

actor_2023 = principals_2023.groupby('nconst').agg(
    total_score=('weighted_score', 'sum'),
    num_movies=('tconst', 'nunique'),
    total_votes=('numVotes', 'sum')
).reset_index().merge(name_basics[['nconst', 'primaryName']], on='nconst', how='left')

actor_2023 = actor_2023.sort_values('total_score', ascending=False)
print(f'Top 15 Most Popular Actors in 2023:')
display(actor_2023.head(15)[['primaryName', 'total_score', 'num_movies', 'total_votes']])